# ProductPilot AI

ProductPilot AI is a multi-agent assistant for product managers. Given one product problem, it analyzes customer feedback, looks at competitors, ranks possible features, asks a human for approval, and then writes a PRD, a sprint plan and an analytics report for the approved feature.

The demo product is StudyFlow, a fictional student productivity app. All datasets are synthetic.

Agents and tools:

1. Feedback Analysis Agent: read_customer_feedback, get_feedback_statistics
2. Competitor Research Agent: get_competitor_data
3. Feature Prioritization Agent: calculate_feature_priority
4. Human approval checkpoint (approve, reject or request changes)
5. PRD Writer Agent: save_prd
6. Sprint Planner Agent: read_approved_prd
7. Product Analytics Agent: calculate_product_metrics

Priority score = 0.40 x Demand + 0.25 x Competitor gap + 0.20 x Business value + 0.15 x Implementation ease

How to run:
1. Add your Gemini key in Colab Secrets as GEMINI_API_KEY and turn on notebook access.
2. Run all cells (Runtime > Run all).
3. Type your decision when the approval prompt appears.
4. If the Gemini quota runs out, run the Resume cell later. Finished stages are not repeated.

## Step 1: Install dependencies

In [1]:
%pip install -q openai-agents pandas

## Step 2: Load the datasets

The notebook needs four CSV files. If they are not found (for example after a Colab runtime reset), the project repository is cloned from GitHub. Existing files are never overwritten.

In [2]:
import os
import subprocess
from pathlib import Path

import pandas as pd

GITHUB_REPO_URL = "https://github.com/simranbisht1/ProductPilot-AI.git"

DATASET_FILES = [
    "customer_feedback.csv",
    "competitor_data.csv",
    "engineering_estimates.csv",
    "product_analytics.csv",
]

EXPECTED_COLUMNS = {
    "customer_feedback.csv": ["feedback_id", "feedback_text", "category"],
    "competitor_data.csv": [
        "product", "deadline_reminders", "calendar", "dark_mode",
        "study_tracking", "cross_device_sync", "performance_rating",
    ],
    "engineering_estimates.csv": ["feature", "estimated_effort_days", "business_value"],
    "product_analytics.csv": [
        "week", "active_users", "tasks_created", "tasks_completed",
        "missed_deadlines", "average_study_hours", "retention_rate",
    ],
}

BASE_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
DEFAULT_PROJECT_DIR = BASE_DIR / "ProductPilot-AI"

# Places where the datasets may already exist, checked in this order.
CANDIDATE_DIRS = [
    DEFAULT_PROJECT_DIR,
    Path.cwd(),
    BASE_DIR,
    Path("/content/drive/MyDrive/ProductPilot-AI"),  # used only if Google Drive is mounted
]


def has_all_datasets(folder: Path) -> bool:
    return all((folder / name).is_file() for name in DATASET_FILES)


PROJECT_DIR = next((d.resolve() for d in CANDIDATE_DIRS if has_all_datasets(d)), None)

if PROJECT_DIR is None:
    if DEFAULT_PROJECT_DIR.exists():
        missing = [n for n in DATASET_FILES if not (DEFAULT_PROJECT_DIR / n).is_file()]
        raise FileNotFoundError(
            f"{DEFAULT_PROJECT_DIR} exists but is missing {missing}. "
            "Nothing was overwritten. Rename or remove that folder and rerun this "
            "cell to restore a fresh copy from GitHub."
        )

    print("Datasets not found in this runtime. Restoring them from GitHub...")
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(DEFAULT_PROJECT_DIR)],
        capture_output=True,
        text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError("git clone failed:\n" + clone.stderr)

    PROJECT_DIR = DEFAULT_PROJECT_DIR.resolve()

if not has_all_datasets(PROJECT_DIR):
    raise FileNotFoundError(f"Datasets are still missing in {PROJECT_DIR}.")

os.chdir(PROJECT_DIR)

print("Project folder:", PROJECT_DIR)
print()

for filename in DATASET_FILES:
    df = pd.read_csv(PROJECT_DIR / filename)
    missing_cols = [c for c in EXPECTED_COLUMNS[filename] if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{filename} is missing columns: {missing_cols}")

    print(f"✅ {filename}: {len(df)} rows, columns OK")

Datasets not found in this runtime. Restoring them from GitHub...
Project folder: /content/ProductPilot-AI

✅ customer_feedback.csv: 15 rows, columns OK
✅ competitor_data.csv: 4 rows, columns OK
✅ engineering_estimates.csv: 7 rows, columns OK
✅ product_analytics.csv: 4 rows, columns OK


## Step 3: Connect to Gemini

Gemini 2.5 Flash Lite is used through Google's OpenAI-compatible endpoint. The API key is read from Colab Secrets.

In [3]:
from google.colab import userdata

from agents import AsyncOpenAI, OpenAIChatCompletionsModel, set_tracing_disabled

try:
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Could not read GEMINI_API_KEY from Colab Secrets. Open the 🔑 Secrets "
        "panel, add GEMINI_API_KEY, and switch on 'Notebook access'."
    ) from error

if not gemini_key:
    raise RuntimeError("GEMINI_API_KEY is empty in Colab Secrets.")

client = AsyncOpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=3,
)

model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash-lite",
    openai_client=client,
)

set_tracing_disabled(True)

print("Gemini configured: gemini-2.5-flash-lite (key loaded from Colab Secrets)")

Gemini configured: gemini-2.5-flash-lite (key loaded from Colab Secrets)


## Step 4: Helper functions

This cell has the shared helpers: file paths, saving and loading the workflow state, and run_agent_verified.

I added run_agent_verified because the agents sometimes skipped their tools or gave incomplete answers. After each agent run it checks which tools were actually called and whether the output is complete. If not, it retries once and tells the agent what was wrong. It also waits and retries when Gemini hits a rate limit.

In [4]:
import asyncio
import json
import re
import shutil
import uuid
from datetime import datetime, timezone

import openai
from agents import ModelSettings, Runner
from agents.exceptions import MaxTurnsExceeded, ModelBehaviorError

OUTPUT_DIR = PROJECT_DIR / "productpilot_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = OUTPUT_DIR / "workflow_state.json"
ARCHIVE_DIR = OUTPUT_DIR / "archive"

# Low temperature keeps tool use and numbers consistent.
AGENT_SETTINGS = ModelSettings(temperature=0.2, max_tokens=8192)

PACE_SECONDS = 4             # short pause between model calls (free-tier friendly)
MAX_ATTEMPTS_PER_STAGE = 2   # 1 normal attempt + 1 corrective retry
MAX_RATE_LIMIT_RETRIES = 3
MAX_REPORT_CHARS = 60_000    # guards against degenerate (e.g. whitespace-padded) output


class StageFailedError(RuntimeError):
    """An agent stage could not produce verified output."""


class QuotaExhaustedError(RuntimeError):
    """Gemini quota is used up; the workflow can be resumed later."""


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def make_safe_filename(feature_name: str) -> str:
    """Create a consistent filename for every feature."""
    return "".join(
        char if char.isalnum() else "_" for char in feature_name.lower()
    ).strip("_")


def report_paths(feature_name: str) -> dict:
    safe_name = make_safe_filename(feature_name)
    return {
        "prd": OUTPUT_DIR / f"{safe_name}_prd.md",
        "sprint_plan": OUTPUT_DIR / f"{safe_name}_sprint_plan.md",
        "analytics_report": OUTPUT_DIR / f"{safe_name}_analytics_report.md",
    }


def read_workflow_state():
    """Read the persistent workflow state (None if missing or invalid)."""
    if not STATE_PATH.is_file():
        return None
    try:
        return json.loads(STATE_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return None


def write_workflow_state(state: dict) -> None:
    """Save the workflow state safely (write to a temp file, then replace)."""
    state["updated_at"] = now_iso()
    temp_path = STATE_PATH.with_suffix(".tmp")
    temp_path.write_text(json.dumps(state, indent=2, default=str), encoding="utf-8")
    temp_path.replace(STATE_PATH)


def clean_markdown(text) -> str:
    """Remove trailing spaces, huge space runs and extra blank lines."""
    if text is None:
        return ""
    lines = []
    for line in str(text).replace("\r\n", "\n").split("\n"):
        line = line.rstrip()
        indent = len(line) - len(line.lstrip(" "))
        body = re.sub(r"[ \t]{2,}", " ", line.lstrip(" "))
        lines.append(" " * min(indent, 16) + body)
    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


def mentions_feature(text: str, feature: str) -> bool:
    """True if the text names the feature (allows '&' vs 'and' wording)."""
    lowered = text.lower()
    if feature.lower() in lowered:
        return True
    keywords = [w for w in re.findall(r"[a-z]+", feature.lower()) if len(w) >= 4]
    return bool(keywords) and all(word in lowered for word in keywords)


def _field(obj, key):
    return obj.get(key) if isinstance(obj, dict) else getattr(obj, key, None)


def summarize_tool_activity(result) -> list:
    """List every tool the agent ACTUALLY executed, with its output."""
    names_by_call_id = {}
    activity = []
    for item in result.new_items:
        if item.type == "tool_call_item":
            names_by_call_id[_field(item.raw_item, "call_id")] = _field(item.raw_item, "name")
        elif item.type == "tool_call_output_item":
            call_id = _field(item.raw_item, "call_id")
            activity.append({
                "tool": names_by_call_id.get(call_id, "unknown"),
                "output": str(item.output),
            })
    return activity


def is_tool_error(output: str) -> bool:
    text = output.strip().lower()
    return text.startswith("error") or text.startswith("an error occurred")


def _is_daily_quota_error(message: str) -> bool:
    lowered = message.lower()
    return "perday" in lowered or "per day" in lowered


def _suggested_wait_seconds(message: str, attempt: int) -> float:
    match = re.search(r"retry in ([0-9.]+)s", message, re.IGNORECASE) or re.search(
        r"retryDelay['\"]?\s*:\s*['\"]([0-9.]+)s", message
    )
    wait = float(match.group(1)) + 2 if match else 20 * attempt
    return min(wait, 90)


async def run_agent_once(agent, prompt, max_turns: int = 10):
    """Run an agent, waiting and retrying on temporary Gemini errors."""
    for attempt in range(1, MAX_RATE_LIMIT_RETRIES + 2):
        try:
            return await Runner.run(agent, prompt, max_turns=max_turns)

        except openai.RateLimitError as error:
            message = str(error)
            if _is_daily_quota_error(message):
                raise QuotaExhaustedError(
                    "Gemini's daily free-tier quota is used up."
                ) from error
            if attempt > MAX_RATE_LIMIT_RETRIES:
                raise QuotaExhaustedError(
                    "Gemini is still rate-limiting after several retries."
                ) from error
            wait = _suggested_wait_seconds(message, attempt)
            print(f"   ⏳ Rate limited by Gemini. Waiting {wait:.0f}s, then retrying...")
            await asyncio.sleep(wait)

        except (openai.InternalServerError, openai.APIConnectionError, openai.APITimeoutError) as error:
            if attempt > MAX_RATE_LIMIT_RETRIES:
                raise
            wait = 15 * attempt
            print(f"   ⏳ Temporary Gemini error ({type(error).__name__}). Retrying in {wait}s...")
            await asyncio.sleep(wait)

        except (openai.AuthenticationError, openai.PermissionDeniedError) as error:
            raise RuntimeError(
                "Gemini rejected the API key. Check GEMINI_API_KEY in Colab Secrets."
            ) from error


async def run_agent_verified(
    agent,
    prompt: str,
    *,
    stage: str,
    state: dict,
    required_tools: list = (),
    validate=None,
    max_turns: int = 10,
    allow_empty_output: bool = False,
    max_attempts: int = MAX_ATTEMPTS_PER_STAGE,
    continue_conversation: bool = False,
):
    """Run an agent and verify its REAL tool executions and output.

    Returns (final_output, tool_activity). Raises StageFailedError if the
    output still fails verification after the corrective retry.
    """
    problem_summary = ""
    previous_result = None

    for attempt in range(1, max_attempts + 1):
        attempt_input = prompt
        if attempt > 1:
            correction = (
                "IMPORTANT: your previous attempt was rejected by the verification "
                f"step because: {problem_summary}\n"
                f"You must call these tools: {list(required_tools) or 'none'}, "
                "wait for their results, and follow every instruction exactly."
            )
            if continue_conversation and previous_result is not None:
                # Keep the agent's earlier work so it only has to fix the problem.
                attempt_input = previous_result.to_input_list() + [
                    {"role": "user", "content": correction}
                ]
            else:
                attempt_input = prompt + "\n\n" + correction

        await asyncio.sleep(PACE_SECONDS)

        try:
            result = await run_agent_once(agent, attempt_input, max_turns=max_turns)
        except (MaxTurnsExceeded, ModelBehaviorError) as error:
            problem_summary = f"{type(error).__name__}: {error}"
            print(f"   ⚠️ Attempt {attempt} failed: {problem_summary}")
            continue

        activity = summarize_tool_activity(result)
        called = {entry["tool"] for entry in activity}
        succeeded = {entry["tool"] for entry in activity if not is_tool_error(entry["output"])}

        output = result.final_output
        if isinstance(output, str):
            output = clean_markdown(output)

        problems = []
        for tool_name in required_tools:
            if tool_name not in called:
                problems.append(f"required tool '{tool_name}' was not called")
            elif tool_name not in succeeded:
                errors = [e["output"][:300] for e in activity if e["tool"] == tool_name]
                problems.append(f"tool '{tool_name}' returned an error: {errors[-1]}")

        if isinstance(output, str):
            if not output and not allow_empty_output:
                problems.append("the final response was empty")
            elif len(output) > MAX_REPORT_CHARS:
                problems.append("the final response was abnormally long")
        elif output is None:
            problems.append("no final output was produced")

        if not problems and validate is not None:
            problems.extend(validate(output, activity))

        state.setdefault("tool_log", []).append({
            "stage": stage,
            "agent": agent.name,
            "attempt": attempt,
            "tools_called": [entry["tool"] for entry in activity],
            "verified": not problems,
            "problems": problems,
            "time": now_iso(),
        })
        write_workflow_state(state)

        tools_text = ", ".join(entry["tool"] for entry in activity) or "none"
        if not problems:
            print(f"   ✅ Verified ({agent.name}) — tools executed: {tools_text}")
            return output, activity

        problem_summary = "; ".join(problems)
        previous_result = result
        print(f"   ⚠️ Attempt {attempt} rejected — {problem_summary}")
        preview = " ".join(str(output or "(empty reply)").split())[:200]
        print(f"      Agent replied: {preview}")

    raise StageFailedError(
        f"{stage} failed after {max_attempts} attempts: {problem_summary}"
    )


print("Helpers ready. Outputs folder:", OUTPUT_DIR)

Helpers ready. Outputs folder: /content/ProductPilot-AI/productpilot_outputs


## Step 5: Data validation and calculations

All numbers (priority scores and product metrics) are calculated here in plain Python, not by the model. The tools use these same functions.

The data is checked for missing columns, empty files, duplicate IDs, valid value ranges, and feedback categories that match the feature names in the engineering estimates.

Note: Study Progress Dashboard has no competitor data, so its competitor gap score is 0.

In [5]:
class DataValidationError(ValueError):
    """Raised when a dataset is missing, empty or inconsistent."""


FEEDBACK_PATH = PROJECT_DIR / "customer_feedback.csv"
COMPETITOR_PATH = PROJECT_DIR / "competitor_data.csv"
ENGINEERING_PATH = PROJECT_DIR / "engineering_estimates.csv"
ANALYTICS_PATH = PROJECT_DIR / "product_analytics.csv"

# Weighted scoring model — DO NOT change without a strong reason.
WEIGHTS = {
    "customer_demand": 0.40,
    "competitor_gap": 0.25,
    "business_value": 0.20,
    "implementation_ease": 0.15,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

# Engineering feature name -> competitor_data.csv column
FEATURE_COMPETITOR_COLUMNS = {
    "Deadline Reminders": "deadline_reminders",
    "Calendar": "calendar",
    "Performance & Stability": "performance_rating",
    "Dark Mode": "dark_mode",
    "Study Hour Tracking": "study_tracking",
    "Study Progress Dashboard": None,  # no competitor data -> gap scored 0 (stated in output)
    "Cross-Device Sync": "cross_device_sync",
}
BINARY_COMPETITOR_COLUMNS = [
    "deadline_reminders", "calendar", "dark_mode", "study_tracking", "cross_device_sync",
]


def _read_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise DataValidationError(f"Dataset not found: {path.name}")
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        raise DataValidationError(f"{path.name} is empty.")
    df.columns = [str(col).strip() for col in df.columns]
    if df.empty:
        raise DataValidationError(f"{path.name} contains no data rows.")
    return df


def _require_columns(df, required, name):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise DataValidationError(
            f"{name} is missing required column(s) {missing}. Found: {list(df.columns)}"
        )


def _require_no_missing(df, columns, name):
    missing_counts = df[columns].isna().sum()
    bad = {col: int(n) for col, n in missing_counts.items() if n}
    if bad:
        raise DataValidationError(f"{name} has missing values: {bad}")


def _require_numeric(df, columns, name):
    for col in columns:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.isna().any():
            raise DataValidationError(f"{name}: column '{col}' must be numeric.")
        df[col] = converted


def load_feedback() -> pd.DataFrame:
    df = _read_csv(FEEDBACK_PATH)
    columns = ["feedback_id", "feedback_text", "category"]
    _require_columns(df, columns, "customer_feedback.csv")
    _require_no_missing(df, columns, "customer_feedback.csv")

    df["feedback_text"] = df["feedback_text"].astype(str).str.strip()
    df["category"] = df["category"].astype(str).str.strip()

    if (df["feedback_text"] == "").any() or (df["category"] == "").any():
        raise DataValidationError("customer_feedback.csv has blank feedback text or category.")

    duplicates = df.loc[df["feedback_id"].duplicated(), "feedback_id"].tolist()
    if duplicates:
        raise DataValidationError(f"customer_feedback.csv has duplicate feedback IDs: {duplicates}")

    return df


def load_competitors() -> pd.DataFrame:
    df = _read_csv(COMPETITOR_PATH)
    columns = ["product"] + BINARY_COMPETITOR_COLUMNS + ["performance_rating"]
    _require_columns(df, columns, "competitor_data.csv")
    _require_no_missing(df, columns, "competitor_data.csv")
    _require_numeric(df, BINARY_COMPETITOR_COLUMNS + ["performance_rating"], "competitor_data.csv")

    df["product"] = df["product"].astype(str).str.strip()
    studyflow_rows = (df["product"] == "StudyFlow").sum()
    if studyflow_rows != 1:
        raise DataValidationError(
            f"competitor_data.csv must contain exactly one StudyFlow row (found {studyflow_rows})."
        )
    if len(df) < 2:
        raise DataValidationError("competitor_data.csv has no competitor rows.")

    for col in BINARY_COMPETITOR_COLUMNS:
        if not df[col].isin([0, 1]).all():
            raise DataValidationError(f"competitor_data.csv: '{col}' must contain only 0 or 1.")

    if not df["performance_rating"].between(1, 5).all():
        raise DataValidationError("competitor_data.csv: performance_rating must be between 1 and 5.")

    return df


def load_engineering() -> pd.DataFrame:
    df = _read_csv(ENGINEERING_PATH)
    columns = ["feature", "estimated_effort_days", "business_value"]
    _require_columns(df, columns, "engineering_estimates.csv")
    _require_no_missing(df, columns, "engineering_estimates.csv")
    _require_numeric(df, ["estimated_effort_days", "business_value"], "engineering_estimates.csv")

    df["feature"] = df["feature"].astype(str).str.strip()

    duplicates = df.loc[df["feature"].duplicated(), "feature"].tolist()
    if duplicates:
        raise DataValidationError(f"engineering_estimates.csv has duplicate features: {duplicates}")
    if (df["estimated_effort_days"] <= 0).any():
        raise DataValidationError("engineering_estimates.csv: effort days must be greater than 0.")
    if not df["business_value"].between(1, 10).all():
        raise DataValidationError("engineering_estimates.csv: business_value must be between 1 and 10.")

    return df


def compute_feature_rankings() -> pd.DataFrame:
    """Calculate weighted priority scores for every feature (single source of truth)."""
    feedback = load_feedback()
    competitors = load_competitors()
    engineering = load_engineering()

    # Feedback categories must match engineering feature names,
    # otherwise demand would silently become 0.
    unknown_categories = sorted(set(feedback["category"]) - set(engineering["feature"]))
    if unknown_categories:
        raise DataValidationError(
            f"Feedback categories {unknown_categories} do not match any feature in "
            "engineering_estimates.csv. Feature names must match exactly."
        )

    unmapped = sorted(set(engineering["feature"]) - set(FEATURE_COMPETITOR_COLUMNS))
    if unmapped:
        raise DataValidationError(
            f"No competitor mapping for {unmapped}. Add them to FEATURE_COMPETITOR_COLUMNS."
        )

    demand_counts = feedback["category"].value_counts()
    max_demand = demand_counts.max()
    max_effort = engineering["estimated_effort_days"].max()
    if max_demand <= 0 or max_effort <= 0:
        raise DataValidationError("Maximum demand and maximum effort must be greater than 0.")

    studyflow = competitors[competitors["product"] == "StudyFlow"].iloc[0]
    others = competitors[competitors["product"] != "StudyFlow"]

    rows = []
    for _, row in engineering.iterrows():
        feature = row["feature"]
        effort = float(row["estimated_effort_days"])
        business_value = float(row["business_value"])

        # 1. Customer demand (0-10)
        requests = int(demand_counts.get(feature, 0))
        demand_score = requests / max_demand * 10

        # 2. Competitor gap (0-10)
        column = FEATURE_COMPETITOR_COLUMNS[feature]
        if column is None:
            gap_score = 0.0
            gap_note = "No competitor data for this feature, so its gap is scored 0."
        elif column == "performance_rating":
            competitor_avg = others["performance_rating"].mean()
            gap_score = max(0.0, (competitor_avg - studyflow["performance_rating"]) / 5 * 10)
            gap_note = (
                f"StudyFlow performance rating {studyflow['performance_rating']:.0f}/5 vs "
                f"competitor average {competitor_avg:.2f}/5."
            )
        elif studyflow[column] == 1:
            gap_score = 0.0
            gap_note = "StudyFlow already offers this feature."
        else:
            offering = others.loc[others[column] == 1, "product"].tolist()
            gap_score = others[column].mean() * 10
            gap_note = (
                f"{len(offering)} of {len(others)} competitors offer it "
                f"({', '.join(offering) or 'none'}); StudyFlow does not."
            )

        # 3. Implementation ease (0-10, less effort = easier)
        ease_score = (1 - effort / max_effort) * 10

        priority_score = (
            WEIGHTS["customer_demand"] * demand_score
            + WEIGHTS["competitor_gap"] * gap_score
            + WEIGHTS["business_value"] * business_value
            + WEIGHTS["implementation_ease"] * ease_score
        )

        rows.append({
            "feature": feature,
            "priority_score": round(priority_score, 2),
            "customer_requests": requests,
            "demand_score": round(demand_score, 2),
            "competitor_gap_score": round(gap_score, 2),
            "business_value": business_value,
            "ease_score": round(ease_score, 2),
            "effort_days": effort,
            "competitor_note": gap_note,
        })

    rankings = (
        pd.DataFrame(rows)
        .sort_values(["priority_score", "customer_requests"], ascending=[False, False], kind="stable")
        .reset_index(drop=True)
    )
    rankings.insert(0, "rank", rankings.index + 1)
    rankings.attrs["total_feedback"] = len(feedback)
    return rankings


def compute_product_metrics() -> dict:
    """Validate the analytics dataset and calculate verified product metrics."""
    df = _read_csv(ANALYTICS_PATH)
    columns = [
        "week", "active_users", "tasks_created", "tasks_completed",
        "missed_deadlines", "average_study_hours", "retention_rate",
    ]
    numeric = columns[1:]
    _require_columns(df, columns, "product_analytics.csv")
    _require_no_missing(df, columns, "product_analytics.csv")
    _require_numeric(df, numeric, "product_analytics.csv")

    if (df["tasks_created"] <= 0).any() or (df["active_users"] <= 0).any():
        raise DataValidationError("Tasks created and active users must be positive.")
    if (df["tasks_completed"] > df["tasks_created"]).any():
        raise DataValidationError("Tasks completed cannot exceed tasks created.")
    if (df[["tasks_completed", "missed_deadlines", "average_study_hours"]] < 0).any().any():
        raise DataValidationError("Counts and study hours cannot be negative.")
    if not df["retention_rate"].between(0, 100).all():
        raise DataValidationError("Retention rate must be between 0 and 100.")

    df["completion_rate"] = df["tasks_completed"] / df["tasks_created"] * 100
    df["missed_deadline_rate"] = df["missed_deadlines"] / df["tasks_created"] * 100

    total_tasks = int(df["tasks_created"].sum())
    total_completed = int(df["tasks_completed"].sum())
    total_missed = int(df["missed_deadlines"].sum())
    first_users = df["active_users"].iloc[0]
    last_users = df["active_users"].iloc[-1]

    return {
        "weeks": len(df),
        "first_week": str(df["week"].iloc[0]),
        "last_week": str(df["week"].iloc[-1]),
        "weekly": df[[
            "week", "active_users", "completion_rate", "missed_deadline_rate",
            "retention_rate", "average_study_hours",
        ]].round(2),
        "total_tasks": total_tasks,
        "total_completed": total_completed,
        "total_missed": total_missed,
        "completion_rate": total_completed / total_tasks * 100,
        "missed_deadline_rate": total_missed / total_tasks * 100,
        "user_growth": (last_users - first_users) / first_users * 100,
        "avg_retention": df["retention_rate"].mean(),
        "avg_study_hours": df["average_study_hours"].mean(),
    }


def key_metric_strings(metrics: dict) -> dict:
    """Exact values a correct analytics report must contain."""
    return {
        "completion rate": f"{metrics['completion_rate']:.2f}",
        "missed deadline rate": f"{metrics['missed_deadline_rate']:.2f}",
        "user growth": f"{metrics['user_growth']:.2f}",
    }


print("Data validation and calculation functions ready.")

Data validation and calculation functions ready.


## Step 6: Tools

The seven tools are registered with @function_tool. Each one calls a normal Python function, so the logic can also be tested without calling Gemini. Errors are returned as text starting with "Error:".

In [6]:
from agents import function_tool


# ---------------- TOOL 1 ----------------
def read_customer_feedback_impl() -> str:
    try:
        df = load_feedback()
    except DataValidationError as error:
        return f"Error: {error}"
    except Exception as error:
        return f"Error reading feedback: {error}"

    return (
        "STUDYFLOW CUSTOMER FEEDBACK (synthetic data)\n"
        f"Total records: {len(df)}\n\n"
        + df[["feedback_id", "feedback_text", "category"]].to_string(index=False)
    )


@function_tool
def read_customer_feedback() -> str:
    """Read every StudyFlow customer feedback record (ID, feedback text and category)."""
    return read_customer_feedback_impl()


# ---------------- TOOL 2 ----------------
def get_feedback_statistics_impl() -> str:
    try:
        df = load_feedback()
    except DataValidationError as error:
        return f"Error: {error}"
    except Exception as error:
        return f"Error calculating statistics: {error}"

    counts = df["category"].value_counts()
    total = len(df)
    lines = ["VERIFIED CUSTOMER FEEDBACK STATISTICS (synthetic data)", ""]
    for category, count in counts.items():
        lines.append(f"{category}: {count} of {total} records ({count / total * 100:.1f}%)")
    lines += [
        "",
        f"Total feedback records: {total}",
        f"Distinct categories: {len(counts)}",
        f"Most requested: {counts.index[0]} ({counts.iloc[0]} records)",
    ]
    return "\n".join(lines)


@function_tool
def get_feedback_statistics() -> str:
    """Calculate verified feedback category counts and percentages for StudyFlow."""
    return get_feedback_statistics_impl()


# ---------------- TOOL 3 ----------------
def get_competitor_data_impl() -> str:
    try:
        df = load_competitors()
    except DataValidationError as error:
        return f"Error: {error}"
    except Exception as error:
        return f"Error reading competitor data: {error}"

    studyflow = df[df["product"] == "StudyFlow"].iloc[0]
    others = df[df["product"] != "StudyFlow"]

    lines = [
        "FICTIONAL COMPETITOR DATA (synthetic)",
        "Feature columns: 1 = available, 0 = not available. Performance rating: 1-5.",
        "Note: basic study tracking is not the same as detailed daily study-hour tracking.",
        "",
        df.to_string(index=False),
        "",
        "GAP SUMMARY FOR STUDYFLOW:",
    ]
    for col in BINARY_COMPETITOR_COLUMNS:
        offering = others.loc[others[col] == 1, "product"].tolist()
        status = "has it" if studyflow[col] == 1 else "MISSING"
        lines.append(
            f"- {col}: StudyFlow {status}; offered by {len(offering)} of {len(others)} "
            f"competitors ({', '.join(offering) or 'none'})"
        )
    lines.append(
        f"- performance_rating: StudyFlow {studyflow['performance_rating']:.0f}/5 vs "
        f"competitor average {others['performance_rating'].mean():.2f}/5"
    )
    return "\n".join(lines)


@function_tool
def get_competitor_data() -> str:
    """Read fictional competitor features and performance ratings, with a StudyFlow gap summary."""
    return get_competitor_data_impl()


print("Tools 1-3 created: read_customer_feedback, get_feedback_statistics, get_competitor_data")

Tools 1-3 created: read_customer_feedback, get_feedback_statistics, get_competitor_data


In [7]:
# ---------------- TOOL 4 ----------------
def format_rankings(rankings: pd.DataFrame) -> str:
    lines = []
    for _, r in rankings.iterrows():
        lines.append(
            f"{r['rank']}. {r['feature']} — PRIORITY SCORE {r['priority_score']:.2f} "
            f"(demand {r['demand_score']:.2f} from {r['customer_requests']} requests; "
            f"competitor gap {r['competitor_gap_score']:.2f}; business value {r['business_value']:.0f}; "
            f"ease {r['ease_score']:.2f} for {r['effort_days']:.0f} days) — {r['competitor_note']}"
        )
    return "\n".join(lines)


def calculate_feature_priority_impl() -> str:
    try:
        rankings = compute_feature_rankings()
    except DataValidationError as error:
        return f"Error: {error}"
    except Exception as error:
        return f"Error calculating priorities: {error}"

    top = rankings.iloc[0]
    return (
        "VERIFIED FEATURE PRIORITY RANKING (synthetic planning data)\n"
        "Formula: Priority = 0.40 x Demand + 0.25 x Competitor gap "
        "+ 0.20 x Business value + 0.15 x Implementation ease (all on 0-10 scales)\n\n"
        + format_rankings(rankings)
        + f"\n\nTOP RECOMMENDATION: {top['feature']} (priority score {top['priority_score']:.2f})"
        + "\nNo feature is approved yet. A human must approve the recommendation."
    )


@function_tool
def calculate_feature_priority() -> str:
    """Calculate weighted priority scores (demand 40%, competitor gap 25%, business value 20%, ease 15%) and rank all features."""
    return calculate_feature_priority_impl()


print("Tool 4 created: calculate_feature_priority")

Tool 4 created: calculate_feature_priority


In [8]:
# ---------------- TOOLS 5 & 6 ----------------
PRD_REQUIRED_SECTIONS = {
    "Problem Statement": ["problem"],
    "Functional Requirements": ["functional requirement"],
    "Non-Functional Requirements": ["non-functional", "nonfunctional", "non functional"],
    "User Stories": ["user stor"],
    "Acceptance Criteria": ["acceptance criteria"],
    "Success Metrics": ["success metric", "metrics"],
    "Risks and Dependencies": ["risk"],
}
MIN_PRD_CHARS = 1500


def missing_prd_sections(text: str) -> list:
    lowered = text.lower()
    return [
        section for section, keywords in PRD_REQUIRED_SECTIONS.items()
        if not any(keyword in lowered for keyword in keywords)
    ]


def approval_error(feature_name: str, state):
    """Return an error message if feature_name is not the human-approved feature."""
    if state is None:
        return "Error: Workflow state is missing or invalid."
    approval = state.get("approval") or {}
    if approval.get("status") != "APPROVED":
        return "Error: No feature has been approved by the human reviewer."
    approved = approval.get("approved_feature") or ""
    if not feature_name or make_safe_filename(feature_name) != make_safe_filename(approved):
        return f"Error: '{feature_name}' is not the approved feature. The approved feature is '{approved}'."
    return None


def save_prd_impl(feature_name: str, prd_content: str) -> str:
    state = read_workflow_state()
    error = approval_error(feature_name, state)
    if error:
        return error

    content = clean_markdown(prd_content)
    if not content:
        return "Error: PRD content is empty."
    if len(content) < MIN_PRD_CHARS:
        return (
            f"Error: PRD is too short ({len(content)} characters). Write the complete PRD "
            "with every required section and call save_prd again."
        )
    missing = missing_prd_sections(content)
    if missing:
        return f"Error: PRD is missing required section(s): {missing}. Add them and call save_prd again."

    try:
        path = report_paths(state["approval"]["approved_feature"])["prd"]
        path.write_text(content + "\n", encoding="utf-8")
        if path.read_text(encoding="utf-8").strip() != content:
            return "Error: PRD file could not be verified after saving."
    except Exception as error:
        return f"Error saving PRD: {error}"

    return f"PRD saved successfully: {path.name} ({len(content)} characters)"


def read_approved_prd_impl(feature_name: str) -> str:
    state = read_workflow_state()
    error = approval_error(feature_name, state)
    if error:
        return error

    path = report_paths(state["approval"]["approved_feature"])["prd"]
    if not path.is_file():
        return f"Error: PRD file not found: {path.name}"
    try:
        content = path.read_text(encoding="utf-8")
    except Exception as error:
        return f"Error reading PRD: {error}"
    if not content.strip():
        return "Error: PRD file is empty."
    return content


@function_tool
def save_prd(feature_name: str, prd_content: str) -> str:
    """Save the PRD for the human-approved feature.

    Args:
        feature_name: Exact name of the approved feature.
        prd_content: The complete PRD in Markdown.
    """
    return save_prd_impl(feature_name, prd_content)


@function_tool
def read_approved_prd(feature_name: str) -> str:
    """Read the saved PRD of the human-approved feature.

    Args:
        feature_name: Exact name of the approved feature.
    """
    return read_approved_prd_impl(feature_name)


print("Tools 5-6 created: save_prd, read_approved_prd")

Tools 5-6 created: save_prd, read_approved_prd


In [9]:
# ---------------- TOOL 7 ----------------
def format_product_metrics(m: dict) -> str:
    return f"""STUDYFLOW PRODUCT ANALYTICS (synthetic data, {m['weeks']} weeks)

WEEKLY METRICS:
{m['weekly'].to_string(index=False)}

OVERALL METRICS:
Total tasks created: {m['total_tasks']}
Total tasks completed: {m['total_completed']}
Total missed deadlines: {m['total_missed']}
Overall task completion rate: {m['completion_rate']:.2f}%
Overall missed deadline rate: {m['missed_deadline_rate']:.2f}%
Active user growth ({m['first_week']} to {m['last_week']}): {m['user_growth']:.2f}%
Average weekly retention rate: {m['avg_retention']:.2f}%
Average reported study hours: {m['avg_study_hours']:.2f}

NOTES:
- All data is fictional.
- Rates use tasks created as the denominator.
- Retention is a simple average of weekly percentages, not cohort retention.
- These metrics do not establish causal relationships.
- No new feature has launched during this period.
"""


def calculate_product_metrics_impl() -> str:
    try:
        return format_product_metrics(compute_product_metrics())
    except DataValidationError as error:
        return f"Error: {error}"
    except Exception as error:
        return f"Error calculating product metrics: {error}"


@function_tool
def calculate_product_metrics() -> str:
    """Calculate verified StudyFlow product metrics from the synthetic analytics dataset."""
    return calculate_product_metrics_impl()


ALL_TOOLS = [
    read_customer_feedback, get_feedback_statistics, get_competitor_data,
    calculate_feature_priority, save_prd, read_approved_prd, calculate_product_metrics,
]
print("Tool 7 created: calculate_product_metrics")
print("Total tools:", len(ALL_TOOLS))

Tool 7 created: calculate_product_metrics
Total tools: 7


## Step 7: Agents

The six agents are defined here. The approved feature is passed in when the workflow runs, so the instructions are not tied to a specific feature.

When the human requests changes, the Feature Prioritization Agent reconsiders and returns a structured result (the FeatureRecommendation Pydantic model).

In [10]:
from typing import Literal

from pydantic import BaseModel, Field
from agents import Agent

COMMON_RULES = """
GENERAL RULES:
- StudyFlow is a fictional student productivity app. All data is synthetic.
- Never invent numbers. Use only figures returned by your tools or stated in the request.
- Do not claim live internet research.
- Write clean Markdown. Do NOT use Markdown tables. No long runs of spaces or blank lines.
"""

# ---------------- AGENT 1 ----------------
feedback_agent = Agent(
    name="Feedback Analysis Agent",
    instructions="""
You are the Feedback Analysis Agent for ProductPilot AI.

STEPS:
1. Call read_customer_feedback to read every feedback record.
2. Call get_feedback_statistics to get the verified category counts.
3. Only after both tool results arrive, write a report with these sections:
   ## Main Customer Problems
   ## Verified Feedback Statistics  (copy every category count exactly as returned)
   ## Requested Features
   ## Link to the Product Problem   (how the feedback relates to the problem in the request)
   ## Top 3 Improvement Opportunities

You MUST call BOTH tools. Never recount or estimate frequencies yourself.
""" + COMMON_RULES,
    model=model,
    model_settings=AGENT_SETTINGS,
    tools=[read_customer_feedback, get_feedback_statistics],
)

# ---------------- AGENT 2 ----------------
competitor_agent = Agent(
    name="Competitor Research Agent",
    instructions="""
You are the Competitor Research Agent for ProductPilot AI.

STEPS:
1. Call get_competitor_data.
2. Write a report with these sections:
   ## Feature Comparison
   ## Gaps in StudyFlow
   ## Competitor Strengths
   ## Top 3 Competitive Improvements

A value of 1 means available and 0 means unavailable. Performance ratings range from 1 to 5.
Basic study tracking does not necessarily include detailed study-hour tracking.
All competitors are fictional.
""" + COMMON_RULES,
    model=model,
    model_settings=AGENT_SETTINGS,
    tools=[get_competitor_data],
)

# ---------------- AGENT 3 ----------------
prioritization_agent = Agent(
    name="Feature Prioritization Agent",
    instructions="""
You are the Feature Prioritization Agent for ProductPilot AI.

STEPS:
1. Call calculate_feature_priority.
2. List EVERY feature in the exact order and with the exact scores returned by the tool.
3. Explain why the top 3 features rank highly, using their component scores.
4. Describe important trade-offs.
5. Recommend the feature ranked #1 by the tool, and state its exact priority score.

Weights: customer demand 40%, competitor gap 25%, business value 20%, implementation ease 15%.
Never change scores or the ranking. Do not say any feature is approved; a human decides.
Use beginner-friendly language.
""" + COMMON_RULES,
    model=model,
    model_settings=AGENT_SETTINGS,
    tools=[calculate_feature_priority],
)

# ---------------- AGENT 4 ----------------
prd_agent = Agent(
    name="PRD Writer Agent",
    instructions="""
You are the PRD Writer Agent for ProductPilot AI.

Your ONLY job is to call the save_prd tool with a complete Product Requirements
Document for the human-approved feature named in the request.

CRITICAL: Do NOT write the PRD as a normal chat message. Your FIRST action must be a
save_prd tool call where:
- feature_name = exactly the approved feature name from the request
- prd_content  = the complete PRD in Markdown

The PRD (prd_content) must use these headings:

# <Feature> — Product Requirements Document
## 1. Problem Statement
## 2. Goals and Non-Goals
## 3. Target Users and User Needs
## 4. Proposed Functionality
## 5. Functional Requirements      (numbered FR-1, FR-2, ...)
## 6. Non-Functional Requirements
## 7. User Stories                 ("As a student, I want ... so that ...")
## 8. Acceptance Criteria          (testable; Given/When/Then is fine)
## 9. Success Metrics              (targets are hypothetical - label them)
## 10. Risks and Dependencies
## 11. Assumptions

Keep the PRD focused: roughly 800-1500 words.
If save_prd returns an error, fix the problem and call save_prd again.
After it succeeds, reply with ONE short confirmation sentence (do not repeat the PRD).
Present measured facts only from the request; label everything else as an assumption.
""" + COMMON_RULES,
    model=model,
    model_settings=AGENT_SETTINGS,
    tools=[save_prd],
)

# Gemini sometimes answers with the PRD as plain text instead of calling the tool.
# tool_choice="required" forces a tool call on the first turn (the SDK resets it
# to "auto" afterwards so the agent can finish). Fallback agent = without forcing.
prd_agent_forced = prd_agent.clone(
    model_settings=AGENT_SETTINGS.resolve(ModelSettings(tool_choice="required"))
)

# ---------------- AGENT 5 ----------------
sprint_agent = Agent(
    name="Sprint Planner Agent",
    instructions="""
You are the Sprint Planner Agent for ProductPilot AI.

STEPS:
1. Call read_approved_prd with the approved feature name from the request.
2. Based ONLY on that PRD, write a sprint plan with these sections:
   # <Feature> — Sprint Plan
   ## Sprint Goal
   ## Capacity and Assumptions   (use the engineering estimate given in the request as the
                                  planning constraint; flag clearly if the plan needs more time)
   ## Development Tasks          (for every task: Task ID such as T-1, name, description,
                                  priority High/Medium/Low, suggested role, effort in engineering
                                  days, dependencies, acceptance criteria; include implementation,
                                  testing and integration tasks)
   ## Timeline
   ## Risks and Mitigation
   ## Deliverables
   ## Definition of Done

Your final response must BE the complete sprint plan (not a summary of it).
Never plan unapproved features. Do not claim anything is already implemented.
Distinguish estimates from verified facts.
""" + COMMON_RULES,
    model=model,
    model_settings=AGENT_SETTINGS,
    tools=[read_approved_prd],
)

# ---------------- AGENT 6 ----------------
analytics_agent = Agent(
    name="Product Analytics Agent",
    instructions="""
You are the Product Analytics Agent for ProductPilot AI.

STEPS:
1. Your FIRST action must be to call calculate_product_metrics.
2. Using ONLY the returned figures, write a report with these sections:
   # StudyFlow Product Analytics Report — <Feature>
   ## Data Note                (synthetic data; the approved feature has NOT launched yet)
   ## Verified Metrics         (copy the exact values: overall task completion rate, missed
                               deadline rate, active user growth, average retention, average
                               study hours and the totals)
   ## Trends
   ## Problems Identified
   ## Proposed KPIs for <Feature>   (exactly three; any targets are hypothetical - say so)
   ## Recommended Next Steps

Never invent metrics that the tool did not return (for example MAU, DAU, session length,
feature adoption). Do not invent report dates. Do not make causal claims.
""" + COMMON_RULES,
    model=model,
    model_settings=AGENT_SETTINGS,
    tools=[calculate_product_metrics],
)


# ---------------- STRUCTURED OUTPUT FOR HUMAN-REQUESTED REVISIONS ----------------
class FeatureRecommendation(BaseModel):
    recommended_feature: str = Field(description="Feature recommended after considering the human feedback")
    original_top_feature: str = Field(description="Highest-ranked feature from the verified calculations")
    decision: Literal["KEEP_ORIGINAL", "RECOMMEND_ALTERNATIVE"]
    reasoning: str = Field(description="Explanation of the recommendation")
    tradeoffs: list[str] = Field(description="Important trade-offs considered")
    requires_human_approval: bool = True


# Agent 3 in "revision mode": same role, structured output, no tools needed
# (the verified rankings are passed in the request).
prioritization_revision_agent = Agent(
    name="Feature Prioritization Agent",
    instructions="""
You are the Feature Prioritization Agent for ProductPilot AI, reconsidering a
recommendation because the human product manager requested changes.

Use the verified rankings and the human feedback in the request. Return a structured
recommendation:
1. recommended_feature must be copied exactly from the feature list in the request.
2. original_top_feature must be the #1 feature of the verified ranking.
3. decision = KEEP_ORIGINAL if you recommend the #1 feature, otherwise RECOMMEND_ALTERNATIVE.
4. Never invent or modify priority scores. Distinguish the calculated ranking from product judgement.
5. A lower-ranked feature may be recommended if the human's reasoning justifies it.
6. requires_human_approval must always be true. Never approve anything yourself.
StudyFlow and its data are fictional.
""",
    model=model,
    model_settings=AGENT_SETTINGS,
    output_type=FeatureRecommendation,
)


def validate_feature_recommendation(recommendation: FeatureRecommendation, rankings: list) -> FeatureRecommendation:
    """Check the AI's structured recommendation against the verified rankings."""
    canonical = {make_safe_filename(r["feature"]): r["feature"] for r in rankings}
    original_top = rankings[0]["feature"]

    key = make_safe_filename(recommendation.recommended_feature)
    if key not in canonical:
        raise ValueError(f"AI recommended an unknown feature: {recommendation.recommended_feature!r}")
    if make_safe_filename(recommendation.original_top_feature) != make_safe_filename(original_top):
        raise ValueError("original_top_feature does not match the verified ranking.")

    recommendation.recommended_feature = canonical[key]
    recommendation.original_top_feature = original_top

    expected = "KEEP_ORIGINAL" if recommendation.recommended_feature == original_top else "RECOMMEND_ALTERNATIVE"
    if recommendation.decision != expected:
        raise ValueError(f"decision should be {expected} for this recommendation.")
    if recommendation.requires_human_approval is not True:
        raise ValueError("Human approval cannot be bypassed.")
    return recommendation


SPECIALIZED_AGENTS = [
    feedback_agent, competitor_agent, prioritization_agent,
    prd_agent, sprint_agent, analytics_agent,
]
for agent in SPECIALIZED_AGENTS:
    print(f"✅ {agent.name:<30} tools: {[t.name for t in agent.tools]}")
print("Revision mode:", prioritization_revision_agent.name, "-> structured output FeatureRecommendation")

✅ Feedback Analysis Agent        tools: ['read_customer_feedback', 'get_feedback_statistics']
✅ Competitor Research Agent      tools: ['get_competitor_data']
✅ Feature Prioritization Agent   tools: ['calculate_feature_priority']
✅ PRD Writer Agent               tools: ['save_prd']
✅ Sprint Planner Agent           tools: ['read_approved_prd']
✅ Product Analytics Agent        tools: ['calculate_product_metrics']
Revision mode: Feature Prioritization Agent -> structured output FeatureRecommendation


## Step 8: Offline test

This checks the tools, the priority scores, the approval rules and the recommendation validator without using any API calls.

In [11]:
print("=== OFFLINE SELF-TEST (0 API calls) ===\n")

# 1. Priority scores must match the previously validated results.
rankings_df = compute_feature_rankings()
expected_scores = {
    "Deadline Reminders": 8.34, "Calendar": 6.75, "Performance & Stability": 5.49,
    "Dark Mode": 4.59, "Cross-Device Sync": 4.07, "Study Hour Tracking": 2.75,
    "Study Progress Dashboard": 2.70,
}
for _, row in rankings_df.iterrows():
    assert abs(row["priority_score"] - expected_scores[row["feature"]]) <= 0.01, row["feature"]
print("✅ Priority scores match the validated results:")
print(rankings_df[["rank", "feature", "priority_score", "customer_requests", "effort_days"]].to_string(index=False))

# 2. Every tool returns real data, not an error.
tool_checks = {
    "read_customer_feedback": read_customer_feedback_impl(),
    "get_feedback_statistics": get_feedback_statistics_impl(),
    "get_competitor_data": get_competitor_data_impl(),
    "calculate_feature_priority": calculate_feature_priority_impl(),
    "calculate_product_metrics": calculate_product_metrics_impl(),
}
for name, output in tool_checks.items():
    assert not is_tool_error(output), f"{name}: {output}"
assert "Deadline Reminders: 5 of 15" in tool_checks["get_feedback_statistics"]
assert "TOP RECOMMENDATION: Deadline Reminders (priority score 8.34)" in tool_checks["calculate_feature_priority"]
print("\n✅ Tools 1-4 and 7 return verified data")

# 3. Analytics numbers.
metrics = compute_product_metrics()
assert key_metric_strings(metrics) == {
    "completion rate": "71.37", "missed deadline rate": "7.54", "user growth": "21.43",
}
print("✅ Analytics: completion 71.37%, missed-deadline rate 7.54%, user growth 21.43%")

# 4. Approval guard used by save_prd / read_approved_prd.
approved_state = {"approval": {"status": "APPROVED", "approved_feature": "Deadline Reminders"}}
assert approval_error("Deadline Reminders", None)
assert approval_error("Deadline Reminders", {"approval": {"status": "PENDING"}})
assert approval_error("Calendar", approved_state)
assert approval_error("deadline reminders", approved_state) is None
print("✅ PRD tools refuse unapproved or wrong features")

# 5. PRD section checker and output cleaner.
assert "User Stories" in missing_prd_sections("# PRD\n## Problem Statement")
assert clean_markdown("Plan" + " " * 100000 + "\n\n\n\nEnd") == "Plan\n\nEnd"
print("✅ PRD section check and whitespace cleaner work")

# 6. Structured revision validator.
ranking_records = rankings_df.to_dict("records")
good = FeatureRecommendation(
    recommended_feature="calendar", original_top_feature="Deadline Reminders",
    decision="RECOMMEND_ALTERNATIVE", reasoning="test", tradeoffs=[],
)
assert validate_feature_recommendation(good, ranking_records).recommended_feature == "Calendar"
bad_cases = [
    dict(recommended_feature="Unknown", original_top_feature="Deadline Reminders", decision="RECOMMEND_ALTERNATIVE"),
    dict(recommended_feature="Calendar", original_top_feature="Dark Mode", decision="RECOMMEND_ALTERNATIVE"),
    dict(recommended_feature="Calendar", original_top_feature="Deadline Reminders", decision="KEEP_ORIGINAL"),
    dict(recommended_feature="Calendar", original_top_feature="Deadline Reminders", decision="RECOMMEND_ALTERNATIVE",
         requires_human_approval=False),
]
for case in bad_cases:
    try:
        validate_feature_recommendation(FeatureRecommendation(reasoning="t", tradeoffs=[], **case), ranking_records)
        raise AssertionError(f"Should have been rejected: {case}")
    except ValueError:
        pass
print("✅ Invalid AI recommendations are rejected")

print("\nOFFLINE SELF-TEST PASSED — safe to run the workflow.")

=== OFFLINE SELF-TEST (0 API calls) ===

✅ Priority scores match the validated results:
 rank                  feature  priority_score  customer_requests  effort_days
    1       Deadline Reminders            8.34                  5          5.0
    2                 Calendar            6.75                  3         10.0
    3  Performance & Stability            5.49                  3          7.0
    4                Dark Mode            4.59                  1          3.0
    5        Cross-Device Sync            4.07                  1         12.0
    6      Study Hour Tracking            2.75                  1          6.0
    7 Study Progress Dashboard            2.70                  1          8.0

✅ Tools 1-4 and 7 return verified data
✅ Analytics: completion 71.37%, missed-deadline rate 7.54%, user growth 21.43%
✅ PRD tools refuse unapproved or wrong features
✅ PRD section check and whitespace cleaner work
✅ Invalid AI recommendations are rejected

OFFLINE SELF-TEST PASS

## Step 9: Workflow orchestrator

The orchestrator runs the agents in order and passes each agent's output to the next one. After every stage it saves progress to workflow_state.json.

At the start of a new run, old output files are moved to an archive folder so they can't be mixed up with new results.

If the human requests changes, the Feature Prioritization Agent reconsiders and the human decides again (up to 3 rounds).

In [12]:
STAGE_FILES = {
    "feedback_analysis": "01_feedback_analysis.md",
    "competitor_research": "02_competitor_research.md",
    "feature_prioritization": "03_feature_prioritization.md",
}
MAX_REVISION_ROUNDS = 3


# ---------------- run bookkeeping ----------------
def archive_previous_outputs() -> None:
    """Move files from earlier runs into archive/<timestamp>/ (nothing is deleted)."""
    old_files = [p for p in OUTPUT_DIR.iterdir() if p.is_file()]
    if not old_files:
        return
    target = ARCHIVE_DIR / datetime.now().strftime("%Y%m%d_%H%M%S")
    target.mkdir(parents=True, exist_ok=True)
    for path in old_files:
        shutil.move(str(path), str(target / path.name))
    print(f"Archived {len(old_files)} file(s) from the previous run to {target.relative_to(PROJECT_DIR)}")


def start_new_run(product_problem: str) -> dict:
    archive_previous_outputs()
    state = {
        "project": "ProductPilot AI",
        "product": "StudyFlow",
        "data_type": "synthetic",
        "run_id": uuid.uuid4().hex[:8],
        "started_at": now_iso(),
        "status": "RUNNING",
        "product_problem": product_problem,
        "stages": {},
        "feature_rankings": [],
        "approval": {"status": "PENDING", "approved_feature": None, "review_feedback": None},
        "revisions": [],
        "reports": {},
        "tool_log": [],
        "last_error": None,
    }
    write_workflow_state(state)
    return state


def stage_done(state: dict, stage: str) -> bool:
    return state.get("stages", {}).get(stage, {}).get("status") == "COMPLETED"


def complete_stage(state: dict, stage: str, output_file=None) -> None:
    state.setdefault("stages", {})[stage] = {
        "status": "COMPLETED",
        "completed_at": now_iso(),
        "output_file": str(output_file) if output_file else None,
    }
    write_workflow_state(state)


def save_stage_output(stage: str, text: str) -> Path:
    path = OUTPUT_DIR / STAGE_FILES[stage]
    path.write_text(clean_markdown(text) + "\n", encoding="utf-8")
    return path


def read_stage_output(stage: str, limit: int = 4000) -> str:
    path = OUTPUT_DIR / STAGE_FILES[stage]
    return path.read_text(encoding="utf-8")[:limit] if path.is_file() else "(not available)"


def banner(title: str) -> None:
    print("\n" + "=" * 64)
    print(f"  {title}")
    print("=" * 64)


# ---------------- stages 1-3 ----------------
async def stage_feedback(state: dict) -> None:
    banner("[1/6] Feedback Analysis Agent")
    top_category = load_feedback()["category"].value_counts().index[0]

    def validate(output, activity):
        return [] if mentions_feature(output, top_category) else [
            f"the report does not mention the most requested category '{top_category}'"
        ]

    output, _ = await run_agent_verified(
        feedback_agent,
        f"Product problem:\n{state['product_problem']}\n\n"
        "Analyze all StudyFlow customer feedback in relation to this problem.",
        stage="feedback_analysis", state=state,
        required_tools=["read_customer_feedback", "get_feedback_statistics"],
        validate=validate,
    )
    complete_stage(state, "feedback_analysis", save_stage_output("feedback_analysis", output))


async def stage_competitor(state: dict) -> None:
    banner("[2/6] Competitor Research Agent")
    output, _ = await run_agent_verified(
        competitor_agent,
        f"Product problem:\n{state['product_problem']}\n\n"
        f"Feedback analysis summary:\n{read_stage_output('feedback_analysis')}\n\n"
        "Analyze the fictional competitive landscape for StudyFlow.",
        stage="competitor_research", state=state,
        required_tools=["get_competitor_data"],
    )
    complete_stage(state, "competitor_research", save_stage_output("competitor_research", output))


async def stage_prioritization(state: dict) -> None:
    banner("[3/6] Feature Prioritization Agent")
    rankings = compute_feature_rankings()   # same function the tool uses
    top = rankings.iloc[0]

    def validate(output, activity):
        problems = []
        if not mentions_feature(output, top["feature"]):
            problems.append(f"the report does not name the #1 feature '{top['feature']}'")
        if f"{top['priority_score']:.2f}" not in output:
            problems.append(f"the report does not state the exact top score {top['priority_score']:.2f}")
        return problems

    output, _ = await run_agent_verified(
        prioritization_agent,
        f"Product problem:\n{state['product_problem']}\n\n"
        f"Feedback findings:\n{read_stage_output('feedback_analysis', 2500)}\n\n"
        f"Competitor findings:\n{read_stage_output('competitor_research', 2500)}\n\n"
        "Calculate, rank and explain the StudyFlow feature priorities.",
        stage="feature_prioritization", state=state,
        required_tools=["calculate_feature_priority"],
        validate=validate,
    )

    # The recommendation comes from the verified Python calculation, not from parsing AI text.
    state["feature_rankings"] = json.loads(rankings.to_json(orient="records"))
    state["recommendation"] = {"feature": top["feature"], "priority_score": float(top["priority_score"])}
    complete_stage(state, "feature_prioritization", save_stage_output("feature_prioritization", output))


# ---------------- human approval checkpoint ----------------
def ask_human(question: str, valid: list, scripted: list):
    """Ask until a valid answer is given. `scripted` allows non-interactive test runs."""
    while True:
        if scripted:
            answer = str(scripted.pop(0)).strip()
            print(f"{question}{answer}   (scripted)")
        else:
            answer = input(question).strip()
        if not valid or answer in valid:
            if answer:
                return answer
        print(f"Please enter one of: {', '.join(valid) if valid else 'a non-empty answer'}")


async def run_revision(state: dict, feedback_text: str, current_feature: str) -> FeatureRecommendation:
    rankings = state["feature_rankings"]
    ranking_text = "\n".join(
        f"{r['rank']}. {r['feature']}: priority {r['priority_score']:.2f} "
        f"(requests {r['customer_requests']}, effort {r['effort_days']:.0f} days, "
        f"business value {r['business_value']:.0f}) — {r['competitor_note']}"
        for r in rankings
    )
    prompt = (
        f"Product problem:\n{state['product_problem']}\n\n"
        f"VERIFIED FEATURE RANKING (do not change these scores):\n{ranking_text}\n\n"
        f"Feature currently recommended: {current_feature}\n\n"
        f"HUMAN PRODUCT MANAGER FEEDBACK:\n{feedback_text}\n\n"
        "Reconsider the recommendation and return the structured result."
    )

    def validate(output, activity):
        try:
            validate_feature_recommendation(output, rankings)
            return []
        except ValueError as error:
            return [str(error)]

    recommendation, _ = await run_agent_verified(
        prioritization_revision_agent, prompt,
        stage="revision", state=state, validate=validate, max_turns=3,
    )
    return recommendation


async def human_approval(state: dict, scripted: list) -> bool:
    """Returns True if a feature was approved, False if rejected."""
    banner("✋ HUMAN APPROVAL CHECKPOINT")
    rankings = state["feature_rankings"]
    top = rankings[0]
    score_of = {r["feature"]: r["priority_score"] for r in rankings}
    current = state.get("approval", {}).get("pending_recommendation") or top["feature"]

    while True:
        print("\nVerified ranking:")
        for r in rankings:
            marker = "  ◀ current recommendation" if r["feature"] == current else ""
            print(f"  {r['rank']}. {r['feature']:<26} {r['priority_score']:.2f}{marker}")

        print(f"\nRecommended feature: {current} (priority score {score_of[current]:.2f})")
        options = {"1": f"Approve {current}", "2": "Reject", "3": "Request changes"}
        if current != top["feature"]:
            options["4"] = f"Approve the original top feature ({top['feature']})"
        if len(state["revisions"]) >= MAX_REVISION_ROUNDS:
            options.pop("3")
            print(f"(Maximum of {MAX_REVISION_ROUNDS} revision rounds reached.)")
        for key, label in options.items():
            print(f"  {key}. {label}")

        choice = ask_human("\nEnter your decision: ", list(options), scripted)

        if choice in ("1", "4"):
            feature = current if choice == "1" else top["feature"]
            state["approval"] = {
                "status": "APPROVED",
                "approved_feature": feature,
                "approved_score": score_of[feature],
                "review_feedback": state["revisions"][-1]["feedback"] if state["revisions"] else None,
                "decided_at": now_iso(),
            }
            state["status"] = "RUNNING"
            write_workflow_state(state)
            print(f"\n✅ Approved: {feature}")
            return True

        if choice == "2":
            state["approval"] = {
                "status": "REJECTED", "approved_feature": None,
                "review_feedback": None, "decided_at": now_iso(),
            }
            state["status"] = "STOPPED_REJECTED"
            write_workflow_state(state)
            print("\n❌ Recommendation rejected. No PRD, sprint plan or analytics report will be generated.")
            return False

        # choice == "3": request changes -> AI reconsiders with structured output
        feedback_text = ask_human("Describe the changes you want: ", [], scripted)
        state["approval"] = {
            "status": "CHANGES_REQUESTED", "approved_feature": None,
            "review_feedback": feedback_text, "pending_recommendation": current,
        }
        write_workflow_state(state)

        print("\nFeature Prioritization Agent is reconsidering...")
        recommendation = await run_revision(state, feedback_text, current)
        state["revisions"].append({
            "feedback": feedback_text,
            "ai_recommendation": recommendation.model_dump(),
            "time": now_iso(),
        })
        current = recommendation.recommended_feature
        state["approval"]["pending_recommendation"] = current
        write_workflow_state(state)

        print("\n--- AI REVISION (structured output) ---")
        print("Decision:", recommendation.decision)
        print("Recommended feature:", recommendation.recommended_feature)
        print("Reasoning:", recommendation.reasoning)
        for tradeoff in recommendation.tradeoffs:
            print(" -", tradeoff)


# ---------------- stages 4-6 ----------------
def verified_feature_facts(state: dict, feature: str) -> str:
    r = next(item for item in state["feature_rankings"] if item["feature"] == feature)
    total = len(state["feature_rankings"])
    feedback = load_feedback()
    quotes = feedback.loc[feedback["category"] == feature, "feedback_text"].tolist()
    total_feedback = len(feedback)
    quote_text = "\n".join(f'  - "{q}"' for q in quotes) or "  - (no direct feedback quotes)"
    return (
        f"- Approved feature: {feature}\n"
        f"- Priority score: {r['priority_score']:.2f} (rank {r['rank']} of {total})\n"
        f"- Customer demand: {r['customer_requests']} of {total_feedback} feedback records\n"
        f"- Competitor gap: {r['competitor_note']}\n"
        f"- Business value: {r['business_value']:.0f}/10 (planning assumption)\n"
        f"- Engineering estimate: {r['effort_days']:.0f} engineering days (planning assumption)\n"
        f"- Customer quotes:\n{quote_text}"
    )


async def stage_prd(state: dict) -> None:
    feature = state["approval"]["approved_feature"]
    banner(f"[4/6] PRD Writer Agent — {feature}")
    prd_path = report_paths(feature)["prd"]
    if prd_path.exists():
        prd_path.unlink()          # never reuse a PRD from an earlier attempt
    write_workflow_state(state)    # tools read the approval from disk

    def validate(output, activity):
        saved = [a for a in activity if a["tool"] == "save_prd" and a["output"].startswith("PRD saved successfully")]
        if not saved:
            return ["save_prd never saved the PRD successfully"]
        if not prd_path.is_file():
            return ["the PRD file is not on disk"]
        return []

    prd_prompt = (
        f"Approved feature (use exactly this name for save_prd): {feature}\n"
        f"Approval status: APPROVED by the human product manager\n\n"
        f"Product problem:\n{state['product_problem']}\n\n"
        f"VERIFIED FACTS:\n{verified_feature_facts(state, feature)}\n\n"
        f"Feedback findings:\n{read_stage_output('feedback_analysis', 2500)}\n\n"
        f"Competitor findings:\n{read_stage_output('competitor_research', 2000)}\n\n"
        "Call save_prd now with the complete PRD as prd_content. "
        "Do not write the PRD as a chat message."
    )
    verify_options = dict(
        stage="prd", state=state, required_tools=["save_prd"], validate=validate,
        allow_empty_output=True,     # the PRD itself is verified on disk
        max_attempts=3,
        continue_conversation=True,  # a retry keeps the PRD text already written
    )
    try:
        await run_agent_verified(prd_agent_forced, prd_prompt, **verify_options)
    except openai.BadRequestError as error:
        print(f"   ℹ️ Gemini rejected forced tool use ({str(error)[:120]}). Retrying in normal mode...")
        await run_agent_verified(prd_agent, prd_prompt, **verify_options)
    state["reports"]["prd"] = str(prd_path)
    complete_stage(state, "prd", prd_path)
    print(f"   PRD saved: {prd_path.name} ({prd_path.stat().st_size} bytes)")


async def stage_sprint(state: dict) -> None:
    feature = state["approval"]["approved_feature"]
    effort = next(r["effort_days"] for r in state["feature_rankings"] if r["feature"] == feature)
    banner(f"[5/6] Sprint Planner Agent — {feature}")
    sprint_path = report_paths(feature)["sprint_plan"]
    if sprint_path.exists():
        sprint_path.unlink()

    def validate(output, activity):
        problems = []
        prd_reads = [a for a in activity if a["tool"] == "read_approved_prd" and not is_tool_error(a["output"])]
        if not prd_reads:
            problems.append("the approved PRD was not read successfully")
        if len(output) < 800:
            problems.append(f"the sprint plan is too short ({len(output)} characters)")
        if not mentions_feature(output, feature):
            problems.append(f"the sprint plan does not mention '{feature}'")
        for phrase in ["task", "definition of done"]:
            if phrase not in output.lower():
                problems.append(f"the sprint plan has no '{phrase}' section")
        return problems

    output, _ = await run_agent_verified(
        sprint_agent,
        f"Approved feature: {feature}\n"
        f"Total engineering estimate: {effort:.0f} engineering days (planning constraint)\n\n"
        f"Call read_approved_prd with feature_name=\"{feature}\" and create the sprint plan from it.",
        stage="sprint_plan", state=state, required_tools=["read_approved_prd"], validate=validate,
    )
    sprint_path.write_text(output + "\n", encoding="utf-8")
    state["reports"]["sprint_plan"] = str(sprint_path)
    complete_stage(state, "sprint_plan", sprint_path)
    print(f"   Sprint plan saved: {sprint_path.name} ({sprint_path.stat().st_size} bytes)")


async def stage_analytics(state: dict) -> None:
    feature = state["approval"]["approved_feature"]
    banner(f"[6/6] Product Analytics Agent — {feature}")
    analytics_path = report_paths(feature)["analytics_report"]
    if analytics_path.exists():
        analytics_path.unlink()
    required_values = key_metric_strings(compute_product_metrics())

    def validate(output, activity):
        problems = [
            f"the report is missing the verified {name} ({value}%)"
            for name, value in required_values.items() if value not in output
        ]
        if not mentions_feature(output, feature):
            problems.append(f"the report does not mention '{feature}'")
        return problems

    output, _ = await run_agent_verified(
        analytics_agent,
        f"Approved feature (not launched yet): {feature}\n\n"
        "Call calculate_product_metrics, then write the analytics report with three KPIs "
        f"that would show whether {feature} helps students once it launches.",
        stage="analytics_report", state=state,
        required_tools=["calculate_product_metrics"], validate=validate,
    )
    analytics_path.write_text(output + "\n", encoding="utf-8")
    state["reports"]["analytics_report"] = str(analytics_path)
    complete_stage(state, "analytics_report", analytics_path)
    print(f"   Analytics report saved: {analytics_path.name} ({analytics_path.stat().st_size} bytes)")


# ---------------- final verification ----------------
def verify_final_outputs(state: dict) -> dict:
    banner("FINAL WORKFLOW VERIFICATION")
    feature = state["approval"]["approved_feature"]
    run_started = datetime.fromisoformat(state["started_at"]).timestamp()
    metric_values = key_metric_strings(compute_product_metrics()).values()

    checks = {}
    for name, path in report_paths(feature).items():
        problems = []
        if not path.is_file():
            problems.append("file missing")
        else:
            text = path.read_text(encoding="utf-8")
            if len(text.strip()) < 500:
                problems.append(f"too short ({len(text.strip())} chars)")
            if path.stat().st_mtime < run_started - 1:
                problems.append("file is older than this run (stale)")
            if not mentions_feature(text, feature):
                problems.append(f"does not mention {feature}")
            if name == "prd" and missing_prd_sections(text):
                problems.append(f"missing sections {missing_prd_sections(text)}")
            if name == "analytics_report" and not all(v in text for v in metric_values):
                problems.append("verified metrics missing")
        checks[name] = {"path": str(path), "verified": not problems, "problems": problems}
        print(f"{'✅' if not problems else '❌'} {name:<17} {path.name}  {'; '.join(problems)}")

    # Every agent stage must have a verified tool execution in the log.
    for stage in ["feedback_analysis", "competitor_research", "feature_prioritization",
                  "prd", "sprint_plan", "analytics_report"]:
        ok = any(e["stage"] == stage and e["verified"] for e in state["tool_log"])
        print(f"{'✅' if ok else '❌'} tool execution verified: {stage}")
        if not ok:
            checks[f"tools_{stage}"] = {"verified": False, "problems": ["no verified tool execution"]}

    if not all(c["verified"] for c in checks.values()):
        raise StageFailedError("Final verification failed: " + json.dumps(checks, indent=2))
    return checks


# ---------------- orchestrator ----------------
async def continue_workflow(state: dict, decisions=None) -> dict:
    scripted = list(decisions or [])
    try:
        if not stage_done(state, "feedback_analysis"):
            await stage_feedback(state)
        if not stage_done(state, "competitor_research"):
            await stage_competitor(state)
        if not stage_done(state, "feature_prioritization"):
            await stage_prioritization(state)

        if state["approval"].get("status") != "APPROVED":
            state["status"] = "AWAITING_APPROVAL"
            write_workflow_state(state)
            if not await human_approval(state, scripted):
                return {"status": "REJECTED", "recommended_feature": state["recommendation"]["feature"]}

        if not stage_done(state, "prd"):
            await stage_prd(state)
        if not stage_done(state, "sprint_plan"):
            await stage_sprint(state)
        if not stage_done(state, "analytics_report"):
            await stage_analytics(state)

        verify_final_outputs(state)
        state["status"] = "COMPLETED"
        state["last_error"] = None
        state["completed_at"] = now_iso()
        write_workflow_state(state)

        banner("PRODUCTPILOT AI WORKFLOW COMPLETED ✅")
        return {
            "status": "SUCCESS",
            "approved_feature": state["approval"]["approved_feature"],
            "priority_score": state["approval"]["approved_score"],
            "reports": dict(state["reports"]),
        }

    except QuotaExhaustedError as error:
        state["status"] = "PAUSED_QUOTA"
        state["last_error"] = str(error)
        write_workflow_state(state)
        banner("⏸️  WORKFLOW PAUSED — GEMINI QUOTA")
        print(str(error))
        print("All completed stages are saved. Run the RESUME cell when the quota resets")
        print("(or after adding a new key). Completed stages will not be repeated.")
        return {"status": "PAUSED_QUOTA", "error": str(error)}

    except Exception as error:
        state["status"] = "FAILED"
        state["last_error"] = f"{type(error).__name__}: {error}"
        write_workflow_state(state)
        raise


async def run_productpilot(product_problem: str, decisions=None) -> dict:
    """Start a fresh end-to-end run. `decisions` (optional) scripts the approval answers."""
    if not product_problem or not product_problem.strip():
        raise ValueError("Please provide a product problem.")
    banner("PRODUCTPILOT AI — AI-POWERED PRODUCT MANAGEMENT")
    print("Product problem:", product_problem.strip())
    state = start_new_run(product_problem.strip())
    print("Run ID:", state["run_id"])
    return await continue_workflow(state, decisions)


async def resume_productpilot(decisions=None) -> dict:
    """Continue the last run from its first unfinished stage."""
    state = read_workflow_state()
    if not state or "run_id" not in state:
        print("No saved run found. Start a new run with run_productpilot(...).")
        return {"status": "NO_RUN"}
    if state.get("status") == "COMPLETED":
        print(f"Run {state['run_id']} is already completed. Nothing to resume.")
        return {"status": "SUCCESS", "approved_feature": state["approval"]["approved_feature"],
                "reports": state["reports"]}
    if state.get("status") == "STOPPED_REJECTED":
        print(f"Run {state['run_id']} was rejected by the reviewer. Start a new run instead.")
        return {"status": "REJECTED"}
    print(f"Resuming run {state['run_id']} (status: {state.get('status')})")
    done = [s for s, info in state.get("stages", {}).items() if info.get("status") == "COMPLETED"]
    print("Already completed:", done or "none")
    return await continue_workflow(state, decisions)


print("Orchestrator ready: run_productpilot(...) and resume_productpilot()")

Orchestrator ready: run_productpilot(...) and resume_productpilot()


## Step 10: Run the workflow

A full run uses about 13 to 17 Gemini requests. The approval prompt appears after the first three agents finish.

In [17]:
PRODUCT_PROBLEM = (
    "StudyFlow users are missing deadlines. We need to improve student productivity "
    "and engagement. What should we build next?"
)

productpilot_result = await run_productpilot(PRODUCT_PROBLEM)

print("\nResult:")
print(json.dumps(productpilot_result, indent=2))


  PRODUCTPILOT AI — AI-POWERED PRODUCT MANAGEMENT
Product problem: StudyFlow users are missing deadlines. We need to improve student productivity and engagement. What should we build next?
Archived 7 file(s) from the previous run to productpilot_outputs/archive/20261119_115125
Run ID: 0b15451d

  [1/6] Feedback Analysis Agent
   ✅ Verified (Feedback Analysis Agent) — tools executed: read_customer_feedback, get_feedback_statistics

  [2/6] Competitor Research Agent
   ✅ Verified (Competitor Research Agent) — tools executed: get_competitor_data

  [3/6] Feature Prioritization Agent
   ✅ Verified (Feature Prioritization Agent) — tools executed: calculate_feature_priority

  ✋ HUMAN APPROVAL CHECKPOINT

Verified ranking:
  1. Deadline Reminders         8.34  ◀ current recommendation
  2. Calendar                   6.75
  3. Performance & Stability    5.49
  4. Dark Mode                  4.59
  5. Cross-Device Sync          4.07
  6. Study Hour Tracking        2.75
  7. Study Progress Dash

## Step 11: Resume

Only needed if the previous run stopped because of the Gemini quota. If the last run already finished, this does nothing.

In [ ]:
resume_result = await resume_productpilot()
print(json.dumps(resume_result, indent=2))

## Step 12: View the results

In [18]:
from IPython.display import Markdown, display

state = read_workflow_state() or {}
print("Run ID:          ", state.get("run_id"))
print("Status:          ", state.get("status"))
print("Approval:        ", state.get("approval", {}).get("status"))
print("Approved feature:", state.get("approval", {}).get("approved_feature"))
print("Revision rounds: ", len(state.get("revisions", [])))

print("\nVerified tool executions:")
for entry in state.get("tool_log", []):
    if entry["verified"]:
        print(f"  ✅ {entry['agent']:<30} -> {', '.join(entry['tools_called']) or '(structured output)'}")

print("\nGenerated files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(f"  {path.name:<45} {path.stat().st_size:>7} bytes")

for name, path in state.get("reports", {}).items():
    if Path(path).is_file():
        display(Markdown(f"---\n# 📄 {name.replace('_', ' ').title()}\n"))
        display(Markdown(Path(path).read_text(encoding="utf-8")))

Run ID:           0b15451d
Status:           COMPLETED
Approval:         APPROVED
Approved feature: Calendar
Revision rounds:  3

Verified tool executions:
  ✅ Feedback Analysis Agent        -> read_customer_feedback, get_feedback_statistics
  ✅ Competitor Research Agent      -> get_competitor_data
  ✅ Feature Prioritization Agent   -> calculate_feature_priority
  ✅ Feature Prioritization Agent   -> (structured output)
  ✅ Feature Prioritization Agent   -> (structured output)
  ✅ Feature Prioritization Agent   -> (structured output)
  ✅ PRD Writer Agent               -> save_prd
  ✅ Sprint Planner Agent           -> read_approved_prd
  ✅ Product Analytics Agent        -> calculate_product_metrics

Generated files:
  01_feedback_analysis.md                          2007 bytes
  02_competitor_research.md                        1875 bytes
  03_feature_prioritization.md                     2182 bytes
  calendar_analytics_report.md                     1562 bytes
  calendar_prd.md           

---
# 📄 Prd


# Calendar — Product Requirements Document

## 1. Problem Statement

StudyFlow users are missing deadlines, leading to decreased productivity and engagement. The current application lacks a centralized view for managing academic commitments, making it difficult for students to organize their schedules effectively and keep track of important dates such as assignments and exams.

## 2. Goals and Non-Goals

**Goals:**
* Provide students with a clear, visual overview of their academic deadlines and schedule.
* Improve students' ability to manage their time and avoid missing important due dates.
* Increase user engagement and productivity within the StudyFlow application.
* Address a key customer request and a competitive gap.

**Non-Goals:**
* This feature will not include advanced scheduling functionalities like recurring events or integration with external non-academic calendars (e.g., Google Calendar, Outlook Calendar) in the initial release.
* This feature will not include real-time collaboration or sharing of calendar events.
* This feature will not include time-blocking or detailed study session planning beyond assignment/exam dates.

## 3. Target Users and User Needs

**Target Users:**
* All StudyFlow users, primarily students at the high school and university level.

**User Needs:**
* A centralized place to view all upcoming assignments and exams.
* A way to visualize their weekly and monthly academic schedule.
* To reduce the stress and anxiety associated with missing deadlines.
* To better organize their study time around key academic events.

## 4. Proposed Functionality

The Calendar feature will introduce a dedicated calendar view within StudyFlow. Users will be able to see their assignments and exams plotted on a calendar. This view will serve as a central hub for academic time management, allowing students to anticipate upcoming deadlines and plan their study efforts accordingly. The feature aims to directly address the problem of missed deadlines by providing better visibility and organization.

## 5. Functional Requirements

FR-1: The system shall display a calendar view, accessible from the main navigation.
FR-2: The calendar shall display all user-added assignments with their due dates.
FR-3: The calendar shall display all user-added exams with their dates.
FR-4: Users shall be able to add new assignments with a title, description, and due date.
FR-5: Users shall be able to add new exams with a title, description, and date.
FR-6: Users shall be able to edit existing assignment and exam entries.
FR-7: Users shall be able to delete existing assignment and exam entries.
FR-8: The calendar view shall default to a monthly view, with options to switch to weekly and daily views.
FR-9: Due dates and exam dates shall be visually distinct on the calendar (e.g., different colors or icons).
FR-10: A list of upcoming deadlines (assignments and exams) shall be displayed alongside the calendar view, sorted by date.

## 6. Non-Functional Requirements

* **Performance:** The calendar view should load within 2 seconds. Adding, editing, or deleting events should feel instantaneous (<500ms).
* **Usability:** The interface should be intuitive and easy to navigate, requiring minimal user training.
* **Reliability:** The feature should be stable and not contribute to application crashes or data loss.
* **Scalability:** The system should handle a large number of events per user without performance degradation (assumption: up to 500 events per user).

## 7. User Stories

* As a student, I want to see all my upcoming assignments and exams on a calendar so that I can visualize my workload and plan my study time effectively.
* As a student, I want to be able to add new assignments and exams to the calendar so that I can keep all my academic commitments in one place.
* As a student, I want to be able to edit or delete existing entries on the calendar so that I can keep my schedule up-to-date.
* As a student, I want to see a list of upcoming deadlines alongside the calendar so that I can quickly identify what needs my immediate attention.

## 8. Acceptance Criteria

* **Given** a user has added multiple assignments and exams with different due dates,
    **When** the user navigates to the Calendar view,
    **Then** all assignments and exams are displayed on their respective dates, and upcoming deadlines are listed chronologically.
* **Given** a user is on the Calendar view,
    **When** they tap the "Add Assignment" button,
    **Then** a form appears allowing them to input assignment details (title, description, due date).
* **Given** an assignment is added to the calendar,
    **When** the user edits the due date,
    **Then** the assignment is repositioned on the calendar to reflect the new due date.
* **Given** a user has added an exam,
    **When** they switch between monthly, weekly, and daily views,
    **Then** the exam is displayed correctly in each respective view.

## 9. Success Metrics

* **Adoption Rate:** 30% of active users utilize the Calendar feature within the first month of launch (hypothetical target).
* **Engagement:** Average number of assignments/exams added per active user of the Calendar feature: 5 per month (hypothetical target).
* **Task Completion:** Reduction in reported missed deadlines by 15% among users who actively use the Calendar feature (hypothetical target).
* **User Satisfaction:** Increase in positive feedback related to organization and time management by 10% (hypothetical target).

## 10. Risks and Dependencies

**Risks:**
* **Low Adoption:** Users may not adopt the feature if it is not intuitive or if they are already using external tools.
* **Data Accuracy:** Inaccurate data entry by users could lead to a misleading calendar.
* **Performance Issues:** If not implemented efficiently, the calendar could become slow, especially with many entries.

**Dependencies:**
* Requires a stable backend to store calendar event data.
* Relies on existing user authentication and data management systems.

## 11. Assumptions

* Users will find value in a dedicated in-app calendar for academic purposes.
* Users will be willing to input their assignment and exam data into StudyFlow.
* The current StudyFlow architecture can support the addition of a new calendar module without significant refactoring.
* The engineering estimate of 10 days is accurate for the initial release of this feature.
* The priority score of 6.75 and customer demand of 3/15 feedback records accurately reflect the importance of this feature.
* The competitor analysis highlighting calendar features in TaskMaster, EduPlanner, and FocusMate is accurate and relevant.


---
# 📄 Sprint Plan


# Calendar — Sprint Plan
## Sprint Goal
Implement the core Calendar feature, allowing users to view, add, edit, and delete assignments and exams, providing a centralized academic schedule overview.

## Capacity and Assumptions
* **Capacity:** 10 engineering days (planning constraint).
* **Assumptions:**
    * The engineering estimate of 10 days is accurate for the initial release.
    * The StudyFlow architecture can support the new calendar module without significant refactoring.
    * Backend services for data storage are available and stable.
    * The team has the necessary skills for front-end and back-end development, as well as testing.
    * No unforeseen technical blockers will arise.
    * The plan requires exactly 10 days. If any task takes longer, the sprint goal may be at risk.

## Development Tasks
* **Task ID:** T-1
    * **Name:** Setup Calendar Module and Navigation
    * **Description:** Create the basic structure for the Calendar module and integrate it into the main application navigation.
    * **Priority:** High
    * **Suggested Role:** Frontend Engineer
    * **Effort:** 1 engineering day
    * **Dependencies:** None
    * **Acceptance Criteria:** Calendar view is accessible via main navigation. Basic UI shell for the calendar is present.

* **Task ID:** T-2
    * **Name:** Implement Calendar View (Monthly)
    * **Description:** Develop the front-end component for the monthly calendar view, displaying dates.
    * **Priority:** High
    * **Suggested Role:** Frontend Engineer
    * **Effort:** 2 engineering days
    * **Dependencies:** T-1
    * **Acceptance Criteria:** A functional monthly calendar grid is displayed.

* **Task ID:** T-3
    * **Name:** Implement Event Data Model and API Endpoints
    * **Description:** Define the data model for assignments and exams. Create backend API endpoints for creating, reading, updating, and deleting (CRUD) these events.
    * **Priority:** High
    * **Suggested Role:** Backend Engineer
    * **Effort:** 2 engineering days
    * **Dependencies:** None
    * **Acceptance Criteria:** API endpoints for event CRUD operations are defined and functional. Data model is established.

* **Task ID:** T-4
    * **Name:** Integrate Event Display on Calendar
    * **Description:** Connect the front-end calendar view to the backend API to fetch and display existing assignments and exams. Visually distinguish between assignments and exams.
    * **Priority:** High
    * **Suggested Role:** Frontend Engineer
    * **Effort:** 2 engineering days
    * **Dependencies:** T-2, T-3
    * **Acceptance Criteria:** Assignments and exams are displayed on the correct dates in the calendar view. Different visual indicators are used for assignments and exams.

* **Task ID:** T-5
    * **Name:** Implement Add/Edit/Delete Event Forms and Logic
    * **Description:** Create UI forms for adding and editing assignments/exams. Implement the logic to send data to the backend API and update the calendar view. Implement delete functionality.
    * **Priority:** High
    * **Suggested Role:** Frontend Engineer
    * **Effort:** 2 engineering days
    * **Dependencies:** T-3, T-4
    * **Acceptance Criteria:** Users can successfully add, edit, and delete assignments and exams. Calendar updates reflect these changes.

* **Task ID:** T-6
    * **Name:** Implement Upcoming Deadlines List
    * **Description:** Develop a component to display a list of upcoming assignments and exams, sorted by date, alongside the calendar.
    * **Priority:** Medium
    * **Suggested Role:** Frontend Engineer
    * **Effort:** 1 engineering day
    * **Dependencies:** T-4
    * **Acceptance Criteria:** A chronologically sorted list of upcoming deadlines is displayed.

* **Task ID:** T-7
    * **Name:** Unit and Integration Testing
    * **Description:** Write unit tests for backend API endpoints and frontend components. Perform integration testing to ensure seamless data flow between front-end and back-end.
    * **Priority:** Medium
    * **Suggested Role:** QA Engineer / Backend Engineer / Frontend Engineer
    * **Effort:** 1 engineering day
    * **Dependencies:** T-3, T-4, T-5, T-6
    * **Acceptance Criteria:** All critical functionalities pass automated tests. Integration between components is verified.

## Timeline
* **Day 1-2:** T-1 (Setup), T-2 (Calendar View - partial)
* **Day 3-4:** T-3 (Event Data Model & API), T-2 (Calendar View - complete)
* **Day 5-6:** T-4 (Integrate Events)
* **Day 7-8:** T-5 (Add/Edit/Delete Forms & Logic)
* **Day 9:** T-6 (Upcoming Deadlines List), T-7 (Testing - initial)
* **Day 10:** T-7 (Testing - final), Bug fixing, Sprint Review preparation

## Risks and Mitigation
* **Risk:** Backend API development takes longer than estimated.
    * **Mitigation:** Prioritize core CRUD operations. Defer non-essential API features (e.g., advanced filtering) to a later sprint. Frontend can use mock data initially.
* **Risk:** Frontend calendar rendering performance issues with many events.
    * **Mitigation:** Implement virtualization or pagination for calendar events if performance becomes an issue during development. Optimize data fetching.
* **Risk:** Integration issues between frontend and backend.
    * **Mitigation:** Maintain clear API contracts. Conduct frequent integration checks. Pair programming between frontend and backend engineers.
* **Risk:** Scope creep - requests for additional calendar views or features.
    * **Mitigation:** Strictly adhere to the defined functional requirements for this sprint. Log additional requests for future sprints.

## Deliverables
* A functional Calendar module within StudyFlow.
* Ability for users to add, view, edit, and delete assignments and exams.
* Visual representation of academic deadlines on a monthly calendar.
* A list of upcoming deadlines.
* Unit and integration tests covering the implemented features.

## Definition of Done
* All development tasks (T-1 to T-6) are completed.
* All acceptance criteria for each task are met.
* Code is reviewed and merged into the main branch.
* Unit and integration tests (T-7) are written and passing.
* The feature meets the non-functional requirements for performance and usability.
* The feature is successfully demonstrated and meets the sprint goal.
* No critical or high-priority bugs related to the Calendar feature are outstanding.


---
# 📄 Analytics Report


I have received the product metrics. Now I will generate the report.

# StudyFlow Product Analytics Report — Calendar
## Data Note
(synthetic data; the approved feature has NOT launched yet)
## Verified Metrics
Overall task completion rate: 71.37%
Missed deadline rate: 7.54%
Active user growth: 21.43%
Average retention rate: 71.00%
Average study hours: 2.33
Total tasks created: 5640
Total tasks completed: 4025
Total missed deadlines: 425
## Trends
Active users increased by 21.43% over the 4-week period. The overall task completion rate saw a slight increase from 70.00% to 72.44%, while the missed deadline rate remained relatively stable, fluctuating between 7.39% and 7.60%. Average study hours per week also showed a gradual increase from 2.1 to 2.5. Average weekly retention rate increased from 68% to 74%.
## Problems Identified
The data does not indicate any specific problems, as no new features have been launched. The current metrics show a positive trend in user engagement and task completion.
## Proposed KPIs for Calendar
1. **Task Completion Rate:** Increase the overall task completion rate by 5% (hypothetical target).
2. **Missed Deadline Rate:** Decrease the overall missed deadline rate by 10% (hypothetical target).
3. **Average Study Hours:** Increase average study hours per week by 15% (hypothetical target).
## Recommended Next Steps
Monitor the impact of the Calendar feature on the proposed KPIs once it launches. Collect user feedback specifically related to the Calendar feature to understand its usability and effectiveness.


## Step 13: Download the outputs (optional)

In [19]:
from zipfile import ZIP_DEFLATED, ZipFile

zip_path = PROJECT_DIR / "ProductPilot_outputs.zip"
with ZipFile(zip_path, "w", ZIP_DEFLATED) as zip_file:
    for path in OUTPUT_DIR.glob("*"):
        if path.is_file():
            zip_file.write(path, arcname=f"productpilot_outputs/{path.name}")
print("Created:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running in Colab - download skipped.")

Created: /content/ProductPilot-AI/ProductPilot_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>